In [ ]:
import os
import time
import random

# os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HTTP_PROXY'] = 'http://192.168.1.100:10808'
os.environ['HTTPS_PROXY'] = 'http://192.168.1.100:10808'

import torch
import cv2
import numpy as np
from PIL import Image

# ===== SAM =====
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# ===== Inpainting =====
from diffusers import StableDiffusionInpaintPipeline

DEVICE = "cuda"


# ===============================
# 1. 加载 SAM
# ===============================
def load_sam():
    print("[INFO] Loading SAM...")
    sam = sam_model_registry["vit_h"](checkpoint="./cache/sam_vit_h_4b8939.pth")
    sam.to(DEVICE)
    return SamAutomaticMaskGenerator(sam)


# ===============================
# 2. 找平面（最大mask）
# ===============================
def find_plane_mask(mask_generator, image):
    print("[INFO] Running SAM...")
    masks = mask_generator.generate(image)

    largest = max(masks, key=lambda x: x['area'])
    mask = largest['segmentation'].astype(np.uint8) * 255

    print(f"[INFO] Masks: {len(masks)}, Largest: {largest['area']}")
    return mask


# ===============================
# 3. 从平面中选patch
# ===============================
def sample_patch(mask, min_size=128, max_size=256):
    print("[INFO] Sampling patch...")

    ys, xs = np.where(mask > 0)
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()

    for _ in range(50):
        w = random.randint(min_size, max_size)
        h = random.randint(min_size, max_size)

        cx = random.randint(x_min, x_max)
        cy = random.randint(y_min, y_max)

        x1 = max(cx - w//2, 0)
        y1 = max(cy - h//2, 0)
        x2 = min(x1 + w, mask.shape[1])
        y2 = min(y1 + h, mask.shape[0])

        patch = mask[y1:y2, x1:x2]

        if np.mean(patch > 0) > 0.8:
            print(f"[INFO] Patch: ({x1},{y1}) -> ({x2},{y2})")
            return (x1, y1, x2, y2)

    raise RuntimeError("Patch sampling failed")


# ===============================
# 4. 加载 Inpainting 模型
# ===============================
from diffusers import StableDiffusionInpaintPipeline

DEVICE = "cuda"

def load_inpaint():
    print("[INFO] Loading Inpainting model (auto-detect format)...")
    # 不指定 use_safetensors，让 diffusers 自动选择可用的格式
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        "runwayml/stable-diffusion-inpainting",
        cache_dir="./cache",
        torch_dtype=torch.float16
        # 移除 use_safetensors 参数，自动检测
    ).to(DEVICE)
    
    pipe.enable_attention_slicing()
    return pipe

# ===============================
# 5. 执行 Inpainting
# ===============================
def run_inpaint(pipe, image, coords, prompt, ref_image=None):
    print("[INFO] Running inpainting...")

    x1, y1, x2, y2 = coords

    # 转 PIL
    image_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    # 构造mask（白色=需要生成）
    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    mask[y1:y2, x1:x2] = 255
    mask_pil = Image.fromarray(mask)

    # 可选：加入参考图（简单版本：拼prompt）
    if ref_image is not None:
        prompt = prompt + ", similar to reference graffiti"

    result = pipe(
        prompt=prompt,
        image=image_pil,
        mask_image=mask_pil,
        num_inference_steps=30,
        guidance_scale=7.5
    ).images[0]

    return cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR), mask


# ===============================
# 主流程
# ===============================
def process(image_path, ref_path=None):
    print("\n========== START ==========")

    image = cv2.imread(image_path)
    if image is None:
        raise ValueError("Image load failed")

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # ===== SAM =====
    sam = load_sam()
    plane_mask = find_plane_mask(sam, image_rgb)
    cv2.imwrite("plane_mask.png", plane_mask)

    # ===== patch =====
    coords = sample_patch(plane_mask)

    # ===== Inpainting =====
    pipe = load_inpaint()

    prompt = "small realistic graffiti on wall, spray paint, slightly worn"

    ref_image = None
    if ref_path:
        ref_image = Image.open(ref_path)

    result, patch_mask = run_inpaint(
        pipe,
        image,
        coords,
        prompt,
        ref_image
    )

    # ===== 保存 =====
    cv2.imwrite("patch_mask.png", patch_mask)
    cv2.imwrite("result.jpg", result)

    print("[INFO] Saved result.jpg")
    print("========== END ==========")


# ===============================
# 入口
# ===============================
if __name__ == "__main__":
    process("input.jpg")  
    # 带参考图：
    # process("input.jpg", ref_path="graffiti_ref.png")